In [ ]:
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, GridSearchCV
from sklearn.tree import DecisionTreeClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix,accuracy_score

In [ ]:
df = pd.read_csv('bsmote_balanced_dataset.csv')
df.head()

In [ ]:
X = df.drop('CLASS', axis=1)
y = df['CLASS']

In [ ]:
numeric_cols = ['AGE', 'WARD_encoded', 'MONTH_sin', 'MONTH_cos']
categorical_cols = [col for col in X.columns if col not in numeric_cols]

In [ ]:
scaler = StandardScaler()
X_scaled = X.copy()  # Preserve original X
X_scaled[numeric_cols] = scaler.fit_transform(X[numeric_cols])  # Scale only numeric columns

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.2, random_state=42, stratify=y)

In [ ]:
print("Training set shape:", X_train.shape, y_train.shape)
print("Test set shape:", X_test.shape, y_test.shape)

In [ ]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

In [ ]:
dt = DecisionTreeClassifier(random_state=42)
param_grid = {
    'max_depth': [None, 5, 10, 15, 20],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'criterion': ['gini', 'entropy']
}

In [ ]:
grid_search = GridSearchCV(
    estimator=dt,
    param_grid=param_grid,
    cv=skf,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

In [ ]:
print("Best parameters from GridSearchCV:", grid_search.best_params_)
print("Best cross-validation F1-score:", grid_search.best_score_)

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

In [ ]:
y_pred = best_model.predict(X_test)

In [ ]:
round(accuracy_score(y_test, y_pred)*100,2)

In [ ]:
print(classification_report(y_test, y_pred, target_names=['Dengue (0)', 'Malaria (1)']))

In [ ]:
cm = confusion_matrix(y_test, y_pred)
cm

In [ ]:
cm = confusion_matrix(y_test, y_pred)
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Dengue (0)', 'Malaria (1)'], yticklabels=['Dengue (0)', 'Malaria (1)'])
plt.title('Confusion Matrix for Decision Tree')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.tight_layout()
plt.savefig('dt_cm_bfrREsmpl.png', dpi=400)
plt.show()

# **LIME Explanation**

In [ ]:
!pip install lime

In [ ]:
import lime
import lime.lime_tabular
from lime.lime_tabular import LimeTabularExplainer

In [ ]:
lime_explainer = LimeTabularExplainer(
    training_data=X_train.values,
    feature_names=X.columns,
    class_names=['Dengue (0)', 'Malaria (1)'],
    mode='classification',
    random_state=42
)

In [ ]:
instance_idx = 0
instance = X_test.iloc[instance_idx].values
lime_exp = lime_explainer.explain_instance(
    data_row=instance,
    predict_fn=best_model.predict_proba,
    num_features=len(X.columns)  # Include all features to aggregate later
)

In [ ]:
lime_exp.show_in_notebook()

In [ ]:
len(X.columns)

In [ ]:
instance


In [ ]:
plt.figure()
lime_exp.as_pyplot_figure()
plt.title('LIME Explanation for First Instance (DT)')
plt.tight_layout()
plt.savefig('lime_explanation_instance_0_DT.png', dpi=400)
plt.show()

In [ ]:
num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=len(X.columns))

    # Show explanation
    explanation.show_in_notebook()

In [ ]:
num_instances = 10

for i in range(num_instances):
    print(f"\nInstance {i}")
    print("Dataset index:", X_test.index[i])
    print(X_test.iloc[i])


In [ ]:
num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    #explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=5)
    explanation = lime_explainer.explain_instance(instance, best_model.predict_proba, num_features=len(X.columns))
    a = explanation.as_list()
    # print(explanation.as_list())
    print(a)
    print("\n")

In [ ]:
all_explanations = []

num_instances = 10
for i in range(num_instances):
    instance = X_test.iloc[i]
    explanation = lime_explainer.explain_instance(
        instance,
        best_model.predict_proba,
        num_features=len(X.columns)  # ask for all 22
    )
    all_explanations.append(explanation.as_list())

In [ ]:
from collections import defaultdict
import numpy as np

feature_weights = defaultdict(list)

for exp in all_explanations:
    for feature_cond, weight in exp:
        for col in X.columns:
            if col in feature_cond:
                feature_weights[col].append(abs(weight))
                break

In [ ]:
avg_weight_df = pd.DataFrame({
    "Feature": X.columns,
    "Avg_Absolute_Weight": [
        np.mean(feature_weights[col]) if len(feature_weights[col]) > 0 else 0
        for col in X.columns
    ]
}).sort_values(by="Avg_Absolute_Weight", ascending=False)

print(avg_weight_df)


In [ ]:
len(avg_weight_df)


In [ ]:
thresholds = sorted(
    avg_weight_df["Avg_Absolute_Weight"].unique()
)

print("Threshold values:")
for t in thresholds:
    print(t)



In [ ]:
len(thresholds)

In [ ]:
# Fix ONE sample index
sample_index = 0
x_sample = X_test.iloc[sample_index]

comparison = []

for t in thresholds:
    selected_features = avg_weight_df[
        avg_weight_df["Avg_Absolute_Weight"] >= t
    ]["Feature"].tolist()

    comparison.append({
        "Threshold": t,
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

comparison_df = pd.DataFrame(comparison)
print(comparison_df)


In [ ]:
all_samples_comparison = []

num_instances = 10

for i in range(num_instances):
    sample_row = {
        "Sample_Index": i,
        "Threshold_Comparison": []
    }

    for t in thresholds:
        selected_features = avg_weight_df[
            avg_weight_df["Avg_Absolute_Weight"] >= t
        ]["Feature"].tolist()

        sample_row["Threshold_Comparison"].append({
            "Threshold": t,
            "Num_Selected_Features": len(selected_features),
            "Selected_Features": selected_features
        })

    all_samples_comparison.append(sample_row)


In [ ]:
for sample in all_samples_comparison:
    print(f"\n================ Sample {sample['Sample_Index']} ================")

    for item in sample["Threshold_Comparison"]:
        print(f"Threshold: {item['Threshold']}")
        print(f"Num Selected Features: {item['Num_Selected_Features']}")
        print(f"Selected Features: {item['Selected_Features']}")
        print("-" * 60)

In [ ]:
rows = []

for sample in all_samples_comparison:
    for item in sample["Threshold_Comparison"]:
        rows.append({
            "Sample_Index": sample["Sample_Index"],
            "Threshold": item["Threshold"],
            "Num_Selected_Features": item["Num_Selected_Features"],
            "Selected_Features": item["Selected_Features"]
        })

comparison_df = pd.DataFrame(rows)
print(comparison_df)

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score


In [ ]:
results = []

In [ ]:
for _, row in comparison_df.iterrows():

    sample_index = row["Sample_Index"]
    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print(f"Sample {sample_index}")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)

    # Safety check
    if len(selected_features) == 0:
        print("No features selected. Skipping.")
        continue

In [ ]:
sample0_df = comparison_df[comparison_df["Sample_Index"] == 0]

final_results_dt = []


In [ ]:
for _, row in sample0_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n========================================")
    print(f"Threshold = {threshold}")
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

    # -------------------------------
    # 1. Create reduced datasets
    # -------------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    print("\nX_train_selected.head()")
    display(X_train_selected.head())

    # -------------------------------
    # 2. StratifiedKFold + GridSearch (Decision Tree)
    # -------------------------------
    skf_dt = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    dt = DecisionTreeClassifier(random_state=42)

    param_grid_dt = {
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    }

    grid_search_dt = GridSearchCV(
        estimator=dt,
        param_grid=param_grid_dt,
        cv=skf_dt,
        scoring='f1',
        n_jobs=-1
    )

    grid_search_dt.fit(X_train_selected, y_train)

    print("Best parameters from GridSearchCV:", grid_search_dt.best_params_)
    print("Best cross-validation F1-score:", grid_search_dt.best_score_)

    # -------------------------------
    # 3. Best Decision Tree model
    # -------------------------------
    best_model_dt = grid_search_dt.best_estimator_
    best_model_dt.fit(X_train_selected, y_train)

    # -------------------------------
    # 4. Test on FIRST 10 rows
    # -------------------------------
    X_test_subset = X_test_selected.iloc[:10]
    y_test_subset = y_test.iloc[:10]

    y_pred_subset = best_model_dt.predict(X_test_subset)

    acc_subset = round(accuracy_score(y_test_subset, y_pred_subset) * 100, 2)
    print("Subset Accuracy (%):", acc_subset)

    print("\nClassification Report (Subset):")
    print(classification_report(
        y_test_subset, y_pred_subset,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # -------------------------------
    # 5. Confusion Matrix (BLUE-WHITE)
    # -------------------------------
    cm = confusion_matrix(y_test_subset, y_pred_subset)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )
    plt.title(f'Decision Tree Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # -------------------------------
    # 6. Store results
    # -------------------------------
    final_results_dt.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Subset_Accuracy": acc_subset,
        "CV_F1": grid_search_dt.best_score_,
        "Selected_Features": selected_features
    })


In [ ]:
import shap

explainer = shap.TreeExplainer(best_model)

In [ ]:
X_test_subset = X_test.iloc[:50]

In [ ]:
shap_values = explainer(X_test_subset)
shap_values

In [ ]:
feature_names = X_test_subset.columns.tolist()

In [ ]:
shap_values_positive = shap_values[..., 1]

In [ ]:
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_values_positive, X_test_subset, feature_names=feature_names, show=False)
plt.title("SHAP Beeswarm Summary Plot for Positive Class")
plt.tight_layout()
plt.savefig('shap_beeswarm_plot_dt_bfrREsmpl.png', dpi=400, bbox_inches='tight')
plt.show()

In [ ]:
shap_vals_all_classes = shap_values.values
shap_vals_all_classes

In [ ]:
shap_vals_class_0 = shap_vals_all_classes[:, :, 0]
shap_vals_class_1 = shap_vals_all_classes[:, :, 1]

In [ ]:
shap_vals_class_0

In [ ]:
shap_vals_class_1

In [ ]:
import numpy as np

mean_abs_shap_class_0 = np.mean(np.abs(shap_vals_class_0), axis=0)
mean_abs_shap_class_1 = np.mean(np.abs(shap_vals_class_1), axis=0)

In [ ]:
mean_abs_shap_class_0


In [ ]:
mean_abs_shap_class_0


In [ ]:
mean_abs_shap_class_1

In [ ]:
mean_abs_shap_global = (mean_abs_shap_class_0 + mean_abs_shap_class_1) / 2
mean_abs_shap_global

In [ ]:
import numpy as np

# Unique sorted thresholds
shap_thresholds = sorted(np.unique(mean_abs_shap_global))

print("Unique SHAP Thresholds:")
for t in shap_thresholds:
    print(t)


In [ ]:
len(shap_thresholds)

In [ ]:
import numpy as np

# -------------------------------
# Step 1: Feature names
# -------------------------------
feature_names = X_train.columns.tolist()

# -------------------------------
# Step 2: Unique sorted thresholds (Method 1)
# -------------------------------
shap_thresholds = sorted(np.unique(mean_abs_shap_global))

print("SHAP Thresholds:")
for t in shap_thresholds:
    print(t)

# -------------------------------
# Step 3: Select features for each threshold
# -------------------------------
shap_results = []

for t in shap_thresholds:

    t_clean = float(t)   # clean threshold value

    # Compare threshold with SHAP importance values
    mask = mean_abs_shap_global >= t_clean

    # Select feature names based on mask
    selected_features = [str(f) for f in np.array(feature_names)[mask]]

    # Safety check: skip if no feature selected
    if len(selected_features) == 0:
        continue

    # Store results
    shap_results.append({
        "Threshold": t_clean,
        "Num_Selected_Features": len(selected_features),
        "Selected_Features": selected_features
    })

    # Print output
    print("\n===================================")
    print("Threshold:", t_clean)
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

In [ ]:
print("\nIndex → Feature → Mean |SHAP|")
print("-" * 45)

for i, (fname, shap_val) in enumerate(zip(feature_names, mean_abs_shap_global)):
    print(f"{i:2d} → {fname:20s} → {shap_val:.6f}")

In [ ]:
shap_feature_selection_df = pd.DataFrame(shap_results)
shap_feature_selection_df

In [ ]:
final_results_shap_dt = []

for _, row in shap_feature_selection_df.iterrows():

    threshold = row["Threshold"]
    selected_features = row["Selected_Features"]

    print("\n===================================")
    print("TRAINING FOR SHAP THRESHOLD:", threshold)
    print("Selected Features:", selected_features)
    print("Number of Features:", len(selected_features))

    # ----------------------------
    # 1. Reduce datasets
    # ----------------------------
    X_train_selected = X_train[selected_features]
    X_test_selected  = X_test[selected_features]

    # ----------------------------
    # 2. StratifiedKFold + GridSearch (Decision Tree)
    # ----------------------------
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    dt = DecisionTreeClassifier(random_state=42)

    param_grid = {
        'max_depth': [None, 5, 10, 15, 20],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'criterion': ['gini', 'entropy']
    }

    grid = GridSearchCV(
        estimator=dt,
        param_grid=param_grid,
        cv=skf,
        scoring='f1',
        n_jobs=-1
    )

    grid.fit(X_train_selected, y_train)

    print("Best Params:", grid.best_params_)
    print("CV Best F1:", grid.best_score_)

    best_model = grid.best_estimator_

    # ----------------------------
    # 3. TEST ON FIRST 50 SAMPLES
    # ----------------------------
    X_test_subset = X_test_selected.iloc[:50]
    y_test_subset = y_test.iloc[:50]

    y_pred = best_model.predict(X_test_subset)

    acc = accuracy_score(y_test_subset, y_pred)
    f1  = f1_score(y_test_subset, y_pred)

    print("Subset Accuracy:", round(acc * 100, 2))
    print("Subset F1:", round(f1, 4))

    print("\nClassification Report:")
    print(classification_report(
        y_test_subset,
        y_pred,
        target_names=['Dengue (0)', 'Malaria (1)']
    ))

    # ----------------------------
    # 4. Confusion Matrix (BLUE-WHITE)
    # ----------------------------
    cm = confusion_matrix(y_test_subset, y_pred)

    plt.figure(figsize=(6, 5))
    sns.heatmap(
        cm, annot=True, fmt='d', cmap='Blues',
        xticklabels=['Dengue (0)', 'Malaria (1)'],
        yticklabels=['Dengue (0)', 'Malaria (1)']
    )
    plt.title(f'DT SHAP Confusion Matrix (Threshold={round(threshold,4)})')
    plt.xlabel('Predicted')
    plt.ylabel('Actual')
    plt.tight_layout()
    plt.show()

    # ----------------------------
    # 5. Store results
    # ----------------------------
        # ----------------------------
    # 5. Store results
    # ----------------------------
    final_results_shap_dt.append({
        "Threshold": threshold,
        "Num_Features": len(selected_features),
        "Accuracy": acc,
        "F1_Score": f1,
        "Selected_Features": selected_features
    })
